In [1]:
import pandas as pd
import numpy as np
import scipy
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
from pathlib import Path

from celloracle import motif_analysis as ma
import celloracle as co
co.__version__


'0.20.0'

In [2]:
os.chdir('/home/kl467102/thesis/')
dataset_name = "kim23-hm1"
ref_genome = 'hg19'
cicero_output = Path("results/cicero_output")  / dataset_name
celloracle_dir = Path("results/celloracle_output") / dataset_name
peaks_path = cicero_output / 'peak_gene_list.csv'

### 0. genome preparation

In [3]:
genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
print(ref_genome, "installation: ", genome_installation)


hg19 installation:  True


In [4]:
if not genome_installation:
    import genomepy
    genomepy.install_genome(name=ref_genome, provider="UCSC", genomes_dir=None)
else:
    print(ref_genome, "is installed.")


hg19 is installed.


### load data

In [5]:
peaks = pd.read_csv(peaks_path, index_col=0)
peaks.head()


,peak_id,gene_short_name
0,chr10_100027444_100030515,LOXL4
1,chr10_100174489_100175328,PYROXD2
2,chr10_100205811_100206960,HPS1
3,chr10_100205811_100206960,LOC101927278
4,chr10_101088941_101090070,CNNM1


In [6]:
peaks = ma.check_peak_format(peaks, ref_genome, genomes_dir=None)


Peaks before filtering:  19054
Peaks with invalid chr_name:  0
Peaks with invalid length:  4
Peaks after filtering:  19050


## Instantiate TFinfo object and search for TF binding motifs

In [7]:
tfi = ma.TFinfo(peak_data_frame=peaks, 
                ref_genome=ref_genome,
                genomes_dir=None) 


In [8]:
os.makedirs(celloracle_dir, exist_ok=True)

In [9]:
# Scan motifs. !!CAUTION!! This step may take several hours if you have many peaks!
tfi.scan(fpr=0.02, 
         motifs=None,  # If you enter None, default motifs will be loaded.
         verbose=True)


No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-10-11 13:31:48,314 - DEBUG - using background: genome hg19 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 

Motif scan started .. It may take long time.



Scanning:   0%|          | 0/16571 [00:00<?, ? sequences/s]

AttributeError: 'PosixPath' object has no attribute 'endswith'

In [10]:

# Save tfinfo object
hdf5_path = celloracle_dir / "tf.celloracle.tfinfo"
hdf5_path  = str(hdf5_path)
tfi.to_hdf5(file_path=hdf5_path)


In [11]:
tfi.scanned_df.head()


,seqname,motif_id,factors_direct,factors_indirect,score,pos,strand
0,chr10_100027444_100030515,GM.5.0.Mixed.0001,,"EGR1, SRF",8.491542,717,1
1,chr10_100027444_100030515,GM.5.0.Mixed.0001,,"EGR1, SRF",8.369501,1112,-1
2,chr10_100027444_100030515,GM.5.0.Mixed.0001,,"EGR1, SRF",8.280041,1113,-1
3,chr10_100027444_100030515,GM.5.0.Mixed.0001,,"EGR1, SRF",8.280041,1114,-1
4,chr10_100027444_100030515,GM.5.0.Mixed.0001,,"EGR1, SRF",8.280041,1115,-1


## Filtering motifs


In [12]:
# Reset filtering 
tfi.reset_filtering()

# Do filtering
tfi.filter_motifs_by_score(threshold=10)

# Format post-filtering results.
tfi.make_TFinfo_dataframe_and_dictionary(verbose=True)


Filtering finished: 16924140 -> 3187106
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/16547 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/17467 [00:00<?, ?it/s]

  0%|          | 0/1094 [00:00<?, ?it/s]

In [13]:
df = tfi.to_dataframe()
df.head()


,peak_id,gene_short_name,9430076C15RIK,AC002126.6,AC012531.1,AC226150.2,AFP,AHR,AHRR,AIRE,...,ZNF784,ZNF8,ZNF816,ZNF85,ZSCAN10,ZSCAN16,ZSCAN22,ZSCAN26,ZSCAN31,ZSCAN4
0,chr10_100027444_100030515,LOXL4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,chr10_100174489_100175328,PYROXD2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr10_100205811_100206960,HPS1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10_100205811_100206960,LOC101927278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_101088941_101090070,CNNM1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [14]:
df = tfi.to_dataframe()
parquet_path = celloracle_dir / "tfinfo.parquet"
df.to_parquet(parquet_path)


In [15]:
from celloracle.utility import save_as_pickled_object

# If you want, you can save the result as a dictionary as follows.
td = tfi.to_dictionary(dictionary_type="TF2targetgenes")
save_as_pickled_object(td, celloracle_dir / "TFinfo_TF2targetgenes.pickled")


In [16]:
tf_df = df.copy()